In [0]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 06 · Optimización con seguimiento (tracking)

> Spec, plan y tareas: `specs/004-optimizacion/`. Código: `src/proyecto/models/tuning.py` (adaptado del `tuning.py` del
> ejemplo del curso). Informe: `reports/06_optimizacion.md`.

Buscamos los hiperparámetros de **LightGBM** (el finalista del torneo) que **bajan el costo de negocio**: costo esperado
por cliente con la matriz 5 a 1, llamando si P(interesado) > 1/6. Tres cuidados:

1. Se optimiza **solo el finalista** (LightGBM).
2. El **espacio de búsqueda** rodea los valores de partida del torneo y el **presupuesto** (40 ensayos) se fijó antes.
3. La mejor puntuación de una búsqueda es **optimista**: se **comprueba** en clientes que la búsqueda no vio.

**Lo que NO se toca:** el umbral (1/6 sale de la matriz de costos, no se busca), el balanceo de clases (no se usa: movería
las probabilidades) y la prueba (`test.parquet` sigue cerrado).

## 0. Cargar los datos de entrenamiento

In [0]:
%pip install optuna==5.0.0 lightgbm==4.7.0 xgboost==3.4.1

In [0]:
import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

from proyecto.config import ID_COL, PROCESSED_DIR, ROOT, TUNE_FOLDS, TUNE_HOLDOUT, TUNE_SEED, TUNE_TRIALS
from proyecto.models.train import (build_pipeline, candidates, cost_threshold, evaluate_folds, make_folds,
                                   metrics_from_proba, split_xy)
from proyecto.models.tuning import (bootstrap_cost_difference, bootstrap_metric_difference, make_lgbm, tune)

REPORTS = ROOT / "reports"
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas (la prueba NO se abre hoy)")

## 1. Pipeline y métrica (los del torneo)

Mismo pipeline (`build_pipeline` con las 5 variables nuevas), mismas métricas (`metrics_from_proba`) y la misma métrica
principal del torneo: el **costo de negocio**. Los parámetros de **partida** son los de LightGBM en el torneo.

In [0]:
default_lgbm = candidates()["lightgbm"]
START_PARAMS = {k: default_lgbm.get_params()[k] for k in ["n_estimators", "learning_rate", "num_leaves"]}
print(f"umbral de decisión de negocio (fijo): llamar si P(interesado) > {cost_threshold():.4f}")
print("parámetros de partida (torneo):", START_PARAMS, "+ el resto por defecto de LightGBM")

## 2. Conexión con el seguimiento

`configure_mlflow()` usa Databricks si `.env` tiene la conexión; si no (o con `MLFLOW_FORCE_LOCAL=1`), guarda todo en
`mlflow_local.db` y lo dice.

In [0]:
from proyecto.tracking.mlflow_utils import configure_mlflow, set_experiment

with warnings.catch_warnings(record=True) as avisos:   # mostramos por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    donde = configure_mlflow()
for aviso in avisos:
    print("Aviso:", aviso.message)
experimento = set_experiment("venta-cruzada-optimizacion")
print("MLflow:", donde, "| experimento:", experimento)

> **Si ves `local`:** todo queda en `mlflow_local.db` (raíz del repo, ignorado por Git). Para verlo en el navegador:
> `mlflow ui --backend-store-uri sqlite:///mlflow_local.db`.

## 3. Reservar clientes que la búsqueda no verá

Separamos, **dentro del entrenamiento**, el 25 % de los clientes, estratificado por `Response` (semilla 7). Es válido
partir por fila porque cada cliente aparece una sola vez (`id` único). La búsqueda trabaja solo con el 75 % restante.

In [0]:
tune_df, holdout_df = train_test_split(train, test_size=TUNE_HOLDOUT, stratify=train["Response"], random_state=TUNE_SEED)
assert set(tune_df[ID_COL]).isdisjoint(set(holdout_df[ID_COL])), "hay clientes en los dos lados"
X_tune, y_tune = split_xy(tune_df)
X_hold, y_hold = split_xy(holdout_df)
print(f"búsqueda: {len(tune_df):,} clientes ({y_tune.mean():.2%} interesados) · "
      f"reservados: {len(holdout_df):,} clientes ({y_hold.mean():.2%} interesados)")

## 4. Espacio de búsqueda y presupuesto

| Parámetro | Rango | Motivo |
|---|---|---|
| `n_estimators` | 150 – 800 | Partida 300; con tasa baja hacen falta más árboles |
| `learning_rate` | 0,01 – 0,2 (log) | Partida 0,05; importa el orden de magnitud |
| `num_leaves` | 15 – 127 | Partida 31; más hojas = más interacciones (edad × daño × región) |
| `min_child_samples` | 20 – 500 | Hojas grandes evitan sobreajustar a pocos clientes |
| `subsample` | 0,6 – 1,0 | Submuestreo de filas: reduce varianza |
| `colsample_bytree` | 0,5 – 1,0 | Hay pocas variables: no bajar de la mitad |
| `reg_lambda` | 0,001 – 10 (log) | Regularización L2 de las hojas |

Presupuesto: **40 ensayos** con el muestreador bayesiano TPE de Optuna (aprende qué zonas del espacio dan menor costo).
Cada ensayo se evalúa con **3 pliegues estratificados** del 75 % de búsqueda.

In [0]:
folds = make_folds(X_tune, y_tune, n_splits=TUNE_FOLDS)
print(f"presupuesto: {TUNE_TRIALS} ensayos · {len(folds)} pliegues · ~{len(folds[0][1]):,} clientes por pliegue de validación")

## 5. La búsqueda (un ensayo = una ejecución registrada)

Primero medimos los parámetros de **partida** en los mismos 3 pliegues (la referencia). Después, la búsqueda: cada
ensayo queda en MLflow como ejecución **anidada** dentro de `optuna-lightgbm-cost`, con sus parámetros, **todas** las
métricas (costo, PR-AUC, ROC-AUC, captura, recall, precisión, log loss…) y su desviación entre pliegues.
El estudio se guarda en `optuna_06.db`: si se corta, al volver a ejecutar solo corre los ensayos que faltan.

In [0]:
start_table = evaluate_folds(build_pipeline(default_lgbm, use_engineered=True), X_tune, y_tune, folds)
start_cost = start_table["cost"].mean()
print(f"partida: costo {start_cost:.4f} ± {start_table['cost'].std():.4f} · PR-AUC {start_table['pr_auc'].mean():.4f}")

In [0]:
study = tune(X_tune, y_tune, folds, n_trials=TUNE_TRIALS, metric="cost", storage=ROOT / "optuna_06.db")
best = study.best_trial
print(f"mejor ensayo: #{best.number} · costo {best.value:.4f} ± {best.user_attrs['cost_std']:.4f} "
      f"(partida {start_cost:.4f}; diferencia {best.value - start_cost:+.4f})")
print("mejores parámetros:", {k: round(v, 4) if isinstance(v, float) else v for k, v in study.best_params.items()})

In [0]:
costs = [t.value for t in study.trials if t.value is not None]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(costs, "o", color="#9aa0a6", label="ensayo")
ax.plot(np.minimum.accumulate(costs), "-", color="#4c72b0", label="mejor hasta ahora")
ax.axhline(start_cost, color="#dd8452", ls="--", label="parámetros de partida")
ax.set_xlabel("ensayo"); ax.set_ylabel("costo en validación cruzada\n(menor es mejor)"); ax.legend()
plt.tight_layout(); plt.show()

### Los 5 mejores ensayos con sus métricas técnicas

El costo decide, pero miramos también lo que puede esconder: si un ensayo baja el costo llamando a muchos más clientes,
o si empeora la calibración (log loss) o el ordenamiento (PR-AUC).

In [0]:
attrs = ["cost", "cost_std", "contact_rate", "recall", "precision", "pr_auc", "roc_auc", "capture_30", "log_loss", "fit_seconds"]
trials = pd.DataFrame([{"ensayo": t.number, **{a: t.user_attrs.get(a) for a in attrs}, **t.params}
                       for t in study.trials if t.value is not None]).set_index("ensayo")
start_row = pd.DataFrame([{**{a: start_table[a].mean() for a in attrs if a in start_table}, "cost_std": start_table["cost"].std(),
                           **START_PARAMS}], index=["partida"])
pd.concat([start_row, trials.sort_values("cost").head(5)]).round(4)

## ¿Qué parámetros importaron?

La importancia (fANOVA de Optuna) dice qué parámetros movieron el costo dentro de la búsqueda. Con 40 ensayos es una
pista, no una prueba.

In [0]:
importance = pd.Series(optuna.importance.get_param_importances(study), name="importancia")
importance.round(3).to_frame()

## 6. La comprobación honesta

Entrenamos **dos** LightGBM con el 75 % de búsqueda (parámetros de **partida** y **optimizados**) y los evaluamos en
los **clientes reservados**, que ninguno vio. La diferencia de costo lleva un intervalo del 95 % con 1.000 remuestreos
de clientes; como control, también la de PR-AUC (300 remuestreos).

**Regla de decisión escrita ANTES de mirar** (`specs/004-optimizacion/spec.md`): se usan los parámetros optimizados
**solo si todo el intervalo del 95 % de (costo optimizado − costo de partida) está por debajo de cero**. Si cruza el
cero, la mejora no se distingue del ruido y se conservan los de partida. La PR-AUC no decide.

In [0]:
tuned_pipe = build_pipeline(make_lgbm(study.best_params), use_engineered=True).fit(X_tune, y_tune)
start_pipe = build_pipeline(candidates()["lightgbm"], use_engineered=True).fit(X_tune, y_tune)
p_tuned = tuned_pipe.predict_proba(X_hold)[:, 1]
p_start = start_pipe.predict_proba(X_hold)[:, 1]
m_tuned, m_start = metrics_from_proba(y_hold, p_tuned), metrics_from_proba(y_hold, p_start)

cost_diff, cost_low, cost_high = bootstrap_cost_difference(y_hold, p_tuned, p_start, n_boot=1000)
pr_diff, pr_low, pr_high = bootstrap_metric_difference(y_hold, p_tuned, p_start, average_precision_score, n_boot=300)

show = ["cost", "contact_rate", "recall", "precision", "pr_auc", "roc_auc", "capture_30", "log_loss", "brier"]
check = pd.DataFrame({"partida": m_start, "optimizado": m_tuned}).loc[show]
check["diferencia (opt. − partida)"] = check["optimizado"] - check["partida"]
check.round(4)

In [0]:
use_tuned = cost_high < 0
verdict = ("sí: todo el intervalo está bajo cero" if use_tuned else
           "no distinguible del ruido: el intervalo cruza el cero" if cost_low <= 0 <= cost_high else
           "empeora: todo el intervalo está sobre cero")
print(f"costo  optimizado − partida = {cost_diff:+.4f}  IC 95 % [{cost_low:+.4f}, {cost_high:+.4f}]  → {verdict}")
print(f"PR-AUC optimizado − partida = {pr_diff:+.4f}  IC 95 % [{pr_low:+.4f}, {pr_high:+.4f}]  (control, no decide)")
print(f"por cada 10.000 clientes: {cost_diff * 10_000:+.0f} unidades de costo (1 = llamada sin éxito; 5 = interesado perdido)")
print("DECISIÓN:", "se usan los parámetros OPTIMIZADOS" if use_tuned else "se conservan los parámetros de PARTIDA")

**Cómo leer el intervalo:** si todo el intervalo está por debajo de cero, el modelo optimizado cuesta menos con
confianza del 95 %. Si cruza el cero, la mejora **no se distingue del ruido** en estos clientes, aunque el número haya
bajado. El resultado y su interpretación quedan escritos en `reports/06_optimizacion.md`.

## 7. Guardar la decisión

La comprobación también queda en MLflow (ejecución `comprobacion-reservada`), y los parámetros elegidos en
`reports/06_best_params.json` para las etapas 07 y 08.

In [0]:
with mlflow.start_run(run_name="comprobacion-reservada"):
    mlflow.set_tags({"etapa": "06_optimizacion", "modelo": "lightgbm", "tipo": "comprobacion",
                     "decision": "optimizado" if use_tuned else "partida"})
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metrics({f"holdout_{k}_tuned": v for k, v in m_tuned.items()})
    mlflow.log_metrics({f"holdout_{k}_start": v for k, v in m_start.items()})
    mlflow.log_metrics({"holdout_cost_diff": cost_diff, "holdout_cost_diff_low": cost_low, "holdout_cost_diff_high": cost_high,
                        "holdout_pr_auc_diff": pr_diff, "holdout_pr_auc_diff_low": pr_low, "holdout_pr_auc_diff_high": pr_high})
    check_run_id = mlflow.active_run().info.run_id

chosen = study.best_params if use_tuned else {**START_PARAMS}
record = {
    "finalista": "lightgbm",
    "decision": "optimizado" if use_tuned else "partida",
    "parametros": chosen,
    "parametros_optimizados": study.best_params,
    "parametros_partida": START_PARAMS,
    "busqueda": {"metrica": "cost", "ensayos": len(costs), "pliegues": TUNE_FOLDS, "mejor_ensayo": best.number,
                 "costo_cv_mejor": best.value, "costo_cv_partida": float(start_cost)},
    "reservado": {"clientes": int(len(y_hold)), "costo_partida": m_start["cost"], "costo_optimizado": m_tuned["cost"],
                  "diferencia": cost_diff, "intervalo_95": [cost_low, cost_high],
                  "pr_auc_diferencia": pr_diff, "pr_auc_intervalo_95": [pr_low, pr_high]},
    "tracking": donde, "experimento": experimento,
    "mlflow_parent_run_id": study.user_attrs.get("mlflow_parent_run_id"), "mlflow_check_run_id": check_run_id,
}
(REPORTS / "06_best_params.json").write_text(json.dumps(record, indent=2, default=float), encoding="utf-8")
check.to_csv(REPORTS / "06_comprobacion.csv")
print("guardado: reports/06_best_params.json y reports/06_comprobacion.csv")

## 8. Comprobar el seguimiento

Las 5 ejecuciones con menor costo del experimento (deben ser ensayos de esta búsqueda):

In [0]:
runs = mlflow.search_runs(experiment_names=[experimento], filter_string="tags.etapa = '06_optimizacion' and metrics.cost > 0",
                          order_by=["metrics.cost ASC"], max_results=5)
cols = [c for c in ["tags.mlflow.runName", "run_id", "metrics.cost", "metrics.cost_std", "metrics.pr_auc", "metrics.log_loss"]
        if c in runs.columns]
print(f"ensayos registrados en esta búsqueda: {len(costs)}")
runs[cols].round(4)

## Entregable de esta etapa

- [x] **Un ensayo = una ejecución registrada**, con parámetros, métricas y etiquetas.
- [x] El **presupuesto** de ensayos (40) está escrito y se respetó.
- [x] Hay **clientes reservados** (25 %) que la búsqueda nunca vio.
- [x] La **comprobación** (intervalo del 95 % remuestreando clientes) está hecha y la regla de decisión estaba escrita antes.
- [x] `reports/06_optimizacion.md` y `reports/06_best_params.json` completados.